# 09 - Stacked Hybrid (Ridge meta-learner)

Learns how to combine the four base predictions (+ side features) via a Ridge meta-model
trained on **5-fold out-of-fold** predictions (leak-free). **Run notebooks 04-07 first.**

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Setup — imports, base SVD params, side-feature dicts, OOF config

In [3]:
from sklearn.model_selection import KFold
from hybrid_recsys.pipeline.features import load_item_features
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel, _to_surprise
from hybrid_recsys.models.hybrid import StackedHybrid
from surprise import SVD as SurpriseSVD

item_features, movie_index = load_item_features()
svd = SVDModel.load()                       # reuse the tuned hyperparameters
global_mean    = float(train["rating"].mean())
train_item_pop = train.groupby("movieId").size().to_dict()
train_user_cnt = train.groupby("userId").size().to_dict()
train_item_cnt = train_item_pop

N_OOF_FOLDS, OOF_SAMPLE_FRAC = 5, 1.0       # set OOF_SAMPLE_FRAC < 1.0 for a faster run
train_oof = (train.sample(frac=OOF_SAMPLE_FRAC, random_state=RANDOM_STATE)
             if OOF_SAMPLE_FRAC < 1.0 else train).reset_index(drop=True)
kf  = KFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof = np.full((len(train_oof), 4), np.nan)
print(f"OOF rows: {len(train_oof):,}")

OOF rows: 19,936,012


## Generate out-of-fold base predictions (leak-free)

In [4]:
for fi, (tr, va) in enumerate(kf.split(train_oof)):
    print(f"  fold {fi+1}/{N_OOF_FOLDS}", end=" ", flush=True)
    ftr, fva = train_oof.iloc[tr], train_oof.iloc[va]
    fur = ftr.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
    fcb = ContentBasedRecommender(n_neighbors=50).fit(item_features, movie_index)
    fuk = UserKNNModel(k=80, min_k=5).fit(ftr)
    fik = ItemKNNModel(k=80, min_k=5).fit(ftr)
    fsv = SurpriseSVD(**svd.best_params); fsv.fit(_to_surprise(ftr).build_full_trainset())
    for n, row in enumerate(fva.itertuples()):
        ur = fur.get(row.userId, {})
        oof[va[n], 0] = fcb.predict(ur, row.movieId)
        oof[va[n], 1] = fuk.predict(row.userId, row.movieId)
        oof[va[n], 2] = fik.predict(row.userId, row.movieId)
        oof[va[n], 3] = fsv.predict(str(row.userId), str(row.movieId)).est
    print("done")

  fold 1/5 Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
done
  fold 2/5 Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
done
  fold 3/5 Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
done
  fold 4/5 Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
done


## Fit the Ridge meta-model & save

In [5]:
keep = ~np.isnan(oof).any(axis=1)
oof, train_oof = oof[keep], train_oof.loc[keep].reset_index(drop=True)

def meta_features(df, base):
    pop  = np.array([train_item_pop.get(m, 0) for m in df["movieId"]], dtype=float)
    ucnt = np.array([train_user_cnt.get(u, 0) for u in df["userId"]],  dtype=float)
    icnt = np.array([train_item_cnt.get(m, 0) for m in df["movieId"]], dtype=float)
    return np.column_stack([base, pop, ucnt, icnt])

stacked = StackedHybrid(alpha=1.0)
stacked.fit(meta_features(train_oof, oof), train_oof["rating"].to_numpy())
stacked.set_side_features(item_popularity=train_item_pop, user_count=train_user_cnt,
                          item_count=train_item_cnt, global_mean=global_mean)
stacked.save()
print("saved stacked_hybrid.joblib")

saved stacked_hybrid.joblib


## Learned Ridge coefficients

In [6]:
coef = pd.DataFrame({"feature": StackedHybrid.FEATURE_NAMES, "weight": stacked.meta.coef_})
display(coef)
fig = px.bar(coef, x="feature", y="weight", title="Stacked Hybrid - learned Ridge coefficients",
             color="weight", color_continuous_scale="RdBu")
fig.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False)
save_fig(fig, "eval_stacked_coefficients")

,feature,weight
0,pred_content,3.455301e-02
1,pred_user_knn,-2.723785e-03
2,pred_item_knn,3.986308e-01
3,pred_svd,6.110259e-01
4,item_popularity,-6.047065e-08
5,user_rating_count,7.281035e-06
6,item_rating_count,0.000000e+00


## Evaluate — compute & record metrics

In [7]:
cb       = ContentBasedRecommender.load()
user_knn = UserKNNModel.load()
item_knn = ItemKNNModel.load()

def st_predict(u, i):
    base = np.array([cb.predict(user_ratings_map.get(u, {}), i),
                     user_knn.predict(u, i), item_knn.predict(u, i), svd.predict(u, i)], dtype=float)
    return stacked.predict_one(u, i, base)

m, preds = full_metrics(st_predict, test, test_sample, train_val, all_movie_ids,
                        n_negatives=N_NEGATIVES, random_state=RANDOM_STATE)
save_metric("Stacked Hybrid", m)
print(f"RMSE={m['rmse']}  MAE={m['mae']}  F1@10={m['k10']['f1']}")

RMSE=0.8054  MAE=0.6029  F1@10=0.4176


## Ranking metrics @ K

In [8]:
save_fig(ranking_curve(m, "Stacked Hybrid"), "eval_stacked_ranking")

## Takeaway

The coefficients lean on SVD and Item-kNN and zero out the weak User-kNN - the meta-model *learned* whom to trust, which is why it tends to win on both RMSE and ranking.